# Monthly user–day–cell table (H3 level 8)

Reads one **whole month** of raw Locomizer signals from Allas (about 112 GiB for Germany, 95–150 GiB for France) and compresses it into one table
with one row per **user, day and H3 cell**, holding the number of signals in that cell in each hour of the day (`h0` … `h23`).

That table is what the commuting notebook (`commuting_munich.ipynb`) uses to find home and work locations: because the
counts are kept per hour, the night and working-hour windows can be changed there without touching the raw data again.

**Why level 8.** Level-8 hexagons are about 0.74 km² (level 9: 0.1 km²), so small GPS jitter between neighbouring cells no longer
splits a user's home or workplace over several cells. The level-8 cell is computed from the level-9 `gid` column.

**How it works**, batch by batch (same scheme as `extract_day.ipynb`):
1. copy a batch of parquet files from Allas to the node-local disk (`$TMPDIR`)
2. aggregate it with DuckDB to (user, day, cell, hour counts) and write a partial result into scratch
3. after the last batch, merge all partial results into one file and remove the partials

Output: `/scratch/project_2022768/<user>/locomizer/<COUNTRY>_userday_h3_<RES>_<MONTH>/userday.parquet`

| column | meaning |
|---|---|
| `userid` | hashed user id |
| `day`, `dow` | day of month and ISO weekday (1 = Monday) |
| `h3_8` | H3 cell at the chosen resolution |
| `n_total` | signals of that user in that cell on that day |
| `h0` … `h23` | the same, split by hour of day (Berlin local time) |

**How to run.** Needs a compute node with ~32 cores and ~128 GB for roughly 15–30 minutes (about 50 BU per hour of use).
Run all cells in Jupyter at <https://www.roihu.csc.fi> with those resources, or submit as a batch job (last section).

In [ ]:
# ---- parameters -------------------------------------------------------------
import os
COUNTRY     = os.environ.get("LOCO_COUNTRY", "DEU")    # DEU, GBR or FRA; batch jobs override it with the LOCO_COUNTRY environment variable
MONTH       = os.environ.get("LOCO_MONTH", "2024-06")  # YYYY-MM; batch jobs override it with LOCO_MONTH
H3_RES_OUT  = int(os.environ.get("LOCO_H3_RES", "8"))   # output cell resolution (7, 8 or 9; the raw data is level 9); batch jobs override it with LOCO_H3_RES
BATCH_GIB   = int(os.environ.get("LOCO_BATCH_GIB", "8"))   # input downloaded per batch (must fit in $TMPDIR: 20 GiB on shared nodes, 600 GiB on a full node)
LOCAL_WORK  = os.environ.get("LOCO_LOCAL_WORK", "0") == "1"   # 1 = partials, merge chunks and spill on the node-local disk (full node only); 0 = on scratch
ALSO_H3_8   = os.environ.get("LOCO_ALSO_H3_8", "0") == "1"    # 1 = also write the level-8 table derived from a level-9 run (saves a second download)
MAX_BATCHES = int(os.environ.get("LOCO_MAX_BATCHES", "0"))   # 0 = whole month; e.g. 2 = smoke test on the first 2 batches only
MERGE_CHUNKS = 8           # the final merge is done per user-hash chunk; one chunk of a level-9 month must fit in memory (a whole month spills > 80 GiB)


In [ ]:
import os, shutil, subprocess, time
from pathlib import Path
import duckdb

# Allas access goes through the rclone remotes created by `allas-conf -m S3 <project>`; nothing here touches credentials.
COUNTRIES = {
    "DEU": {"project": "project_2014316", "bucket": "profile_signals_DEU", "prefix": "profile_signals_DEU"},
    "GBR": {"project": "project_2014314", "bucket": "profile_signals_GBR", "prefix": "profile_signals_GBR"},
    "FRA": {"project": "project_2014315", "bucket": "profile_signals_FRA", "prefix": "profile_signals_FRA"},
}
SCRATCH = Path(f"/scratch/project_2022768/{os.environ['USER']}/locomizer")
RCLONE  = shutil.which("rclone") or "/appl/soft/manual/general/x86_64/allas-tools/bin/rclone"   # from `module load allas`

def remote_path(country, dataset="signals_md5", month=None):
    c = COUNTRIES[country]
    p = f"s3allas-{c['project']}:{c['bucket']}/{c['prefix']}/{dataset}"
    return f"{p}/{month}" if month else p

def list_parts(country, month, dataset="signals_md5"):
    """[(filename, bytes), ...] for one month of a dataset (read-only listing)."""
    res = subprocess.run([RCLONE, "ls", remote_path(country, dataset, month)], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(res.stderr)
    return sorted((name.strip(), int(size)) for size, name in (l.split(maxsplit=1) for l in res.stdout.splitlines()))

threads = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))
mem_gb  = int(os.environ.get("SLURM_MEM_PER_NODE", "16000")) // 1000
tmp     = Path(os.environ.get("TMPDIR", "/tmp")) / "loco_batch"
out_dir = SCRATCH / f"{COUNTRY}_userday_h3_{H3_RES_OUT}_{MONTH}"; out_dir.mkdir(parents=True, exist_ok=True)
work_dir = (tmp.parent / "work") if LOCAL_WORK else out_dir      # where partials, merge chunks and DuckDB spill files go
partial_dir = work_dir / "partial"
partial_dir.mkdir(parents=True, exist_ok=True)

# H3 parent cell from the level-9 index by bit arithmetic (same result as h3.cell_to_parent, but inside DuckDB):
# set the resolution bits (52-55) to H3_RES_OUT and the unused finer digits (3 bits each) to 7.
digit_mask = sum(7 << (45 - 3 * r) for r in range(H3_RES_OUT + 1, 10))
H3_PARENT = (f"lower(hex(((('0x' || gid)::UBIGINT & ~(15::UBIGINT << 52)) "
             f"| ({H3_RES_OUT}::UBIGINT << 52)) | {digit_mask}::UBIGINT))")
HOUR_COLS = ", ".join(f"count(*) FILTER (WHERE hour = {h})::INTEGER AS h{h}" for h in range(24))
SUM_COLS  = ", ".join(f"sum(h{h})::INTEGER AS h{h}" for h in range(24))

duck_tmp = work_dir / "duck_tmp"    # DuckDB spill files: the level-9 merge exceeds the 20 GiB node-local disk of shared nodes, so on scratch unless LOCAL_WORK
shutil.rmtree(duck_tmp, ignore_errors=True)   # spill files left by an interrupted run count against the scratch quota
con = duckdb.connect()
con.execute(f"SET threads = {threads}; SET memory_limit = '{int(mem_gb*0.8)}GB'; SET temp_directory = '{duck_tmp}';"
            "SET preserve_insertion_order = false; SET enable_progress_bar = false;")
print(f"output -> {out_dir}\nwork dir -> {work_dir}\nthreads = {threads}, memory = {mem_gb} GB, local disk = {tmp}, batches of {BATCH_GIB} GiB")

## 1. What is in the bucket

In [ ]:
parts = list_parts(COUNTRY, MONTH)
total = sum(s for _, s in parts)
print(f"{COUNTRY} signals_md5/{MONTH}: {len(parts)} parquet parts, {total/2**30:.1f} GiB")

batches, cur, cur_size = [], [], 0
for name, size in parts:
    if cur and cur_size + size > BATCH_GIB * 2**30:
        batches.append(cur); cur, cur_size = [], 0
    cur.append(name); cur_size += size
if cur:
    batches.append(cur)
if MAX_BATCHES:
    batches = batches[:MAX_BATCHES]
    print(f"SMOKE TEST: only {len(batches)} batch(es)")
print(f"{len(batches)} batches of up to {BATCH_GIB} GiB")

## 2. Download and aggregate each batch

In [ ]:
t0 = time.time()
for i, batch in enumerate(batches):
    out_file = partial_dir / f"part-{i:04d}.parquet"
    if out_file.exists():
        print(f"batch {i+1}/{len(batches)}: exists, skipping", flush=True)
        continue
    shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    (tmp / "files.txt").write_text("\n".join(batch) + "\n")
    subprocess.run([RCLONE, "copy", remote_path(COUNTRY, "signals_md5", MONTH), str(tmp / "in"),
                    "--files-from", str(tmp / "files.txt"), "--transfers", "8", "--checkers", "8", "--quiet"], check=True)
    partial = str(out_file) + ".partial"
    con.execute(f"""
        COPY (
            SELECT Userid AS userid, day::TINYINT AS day, dow::TINYINT AS dow, {H3_PARENT} AS h3_8,
                   count(*)::INTEGER AS n_total, {HOUR_COLS}
            FROM read_parquet('{tmp}/in/*.parquet')
            WHERE month = {int(MONTH[-2:])}
            GROUP BY ALL
        ) TO '{partial}' (FORMAT parquet, COMPRESSION zstd);
    """)
    os.replace(partial, out_file)
    n = con.execute(f"SELECT count(*) FROM read_parquet('{out_file}')").fetchone()[0]
    print(f"batch {i+1}/{len(batches)}: {len(batch)} files -> {n:,} rows  ({time.time()-t0:.0f} s elapsed)", flush=True)
shutil.rmtree(tmp, ignore_errors=True)

## 3. Merge the partial results

A user's signals for one day are spread over many input files, so the same (user, day, cell) row appears in several partial files.
This sums them into one row and writes the final table. The partial files written by this notebook are removed afterwards.

In [ ]:
final = out_dir / "userday.parquet"
chunk_dir = work_dir / "merged"
shutil.rmtree(chunk_dir, ignore_errors=True); chunk_dir.mkdir()
try:
    for k in range(MERGE_CHUNKS):           # one hash aggregate per user chunk, so that it fits in memory instead of spilling to disk
        con.execute(f"""
            COPY (
                SELECT userid, day, dow, h3_8, sum(n_total)::INTEGER AS n_total, {SUM_COLS}
                FROM read_parquet('{partial_dir}/*.parquet')
                WHERE hash(userid) % {MERGE_CHUNKS} = {k}
                GROUP BY userid, day, dow, h3_8
            ) TO '{chunk_dir}/chunk-{k:02d}.parquet' (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 1000000);
        """)
        print(f"merge chunk {k+1}/{MERGE_CHUNKS}  ({time.time()-t0:.0f} s elapsed)", flush=True)
    con.execute(f"""
        COPY (SELECT * FROM read_parquet('{chunk_dir}/*.parquet'))
        TO '{final}.partial' (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 1000000);
    """)
    os.replace(f"{final}.partial", final)
finally:
    shutil.rmtree(duck_tmp, ignore_errors=True)   # spill files count against the scratch quota, also after a failure
shutil.rmtree(chunk_dir)
for f in partial_dir.glob("part-*.parquet"):
    f.unlink()
partial_dir.rmdir()
print(f"merged in {time.time()-t0:.0f} s total")


## 4. Level-8 table derived from the level-9 run (optional)

With `LOCO_ALSO_H3_8=1` a level-9 run also writes the level-8 table the city notebooks read, by aggregating cells to their parent, so a month is downloaded once.


In [ ]:
if ALSO_H3_8 and H3_RES_OUT == 9:
    out8 = SCRATCH / f"{COUNTRY}_userday_h3_8_{MONTH}"; out8.mkdir(parents=True, exist_ok=True)
    chunk8 = work_dir / "merged8"; shutil.rmtree(chunk8, ignore_errors=True); chunk8.mkdir()
    mask8 = 7 << (45 - 3 * 9)                                   # level-9 digit set to 7, resolution bits set to 8
    PARENT8 = f"lower(hex(((('0x' || h3_8)::UBIGINT & ~(15::UBIGINT << 52)) | (8::UBIGINT << 52)) | {mask8}::UBIGINT))"
    try:
        for k in range(MERGE_CHUNKS):
            con.execute(f"""
                COPY (
                    SELECT userid, day, dow, {PARENT8} AS h3_8, sum(n_total)::INTEGER AS n_total, {SUM_COLS}
                    FROM read_parquet('{final}') WHERE hash(userid) % {MERGE_CHUNKS} = {k}
                    GROUP BY userid, day, dow, {PARENT8}
                ) TO '{chunk8}/chunk-{k:02d}.parquet' (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 1000000);
            """)
            print(f"level-8 chunk {k+1}/{MERGE_CHUNKS}  ({time.time()-t0:.0f} s elapsed)", flush=True)
        final8 = out8 / "userday.parquet"
        con.execute(f"COPY (SELECT * FROM read_parquet('{chunk8}/*.parquet')) TO '{final8}.partial' (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 1000000);")
        os.replace(f"{final8}.partial", final8)
    finally:
        shutil.rmtree(duck_tmp, ignore_errors=True)
    shutil.rmtree(chunk8)
    print(f"level-8 table: {final8.stat().st_size/2**30:.2f} GiB -> {final8}  ({time.time()-t0:.0f} s total)")


## 5. Result


In [ ]:
print(con.execute(f"""
    SELECT count(*) AS rows, count(DISTINCT userid) AS users, count(DISTINCT day) AS days,
           count(DISTINCT h3_8) AS cells, sum(n_total) AS signals
    FROM read_parquet('{final}')""").df().to_string(index=False))
print(f"{final.stat().st_size/2**30:.2f} GiB -> {final}")

## Submit as a batch job

From a terminal on `roihu-cpu.csc.fi` (one line, no script file needed):

```bash
sbatch --account=project_2022768 --partition=small --time=02:00:00 --cpus-per-task=32 --mem=128G --job-name=userday_FRA_2024-05 \
  --export=ALL,LOCO_COUNTRY=FRA,LOCO_MONTH=2024-05 --output=/scratch/project_2022768/%u/locomizer/logs/userday_FRA_2024-05_%j.out \
  --wrap="module load allas python-geo; cd ~/locomizer_signals/pipeline; jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output-dir=/scratch/project_2022768/\$USER/locomizer/logs --output=userday_FRA_2024-05.ipynb userday_month.ipynb"
```

Country and month come from the environment variables, so several months can run at the same time; the executed copy of the notebook (with its printed progress) is written to the logs folder, the notebook itself is left untouched.